In [ ]:
!pip install prophet

In [ ]:
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt

# Carica il file
cpi = pd.read_excel("C:\\Users\\vladb\\OneDrive\\Desktop\\ANALISI DATA\\EPICODE\\MODULO 6 (PROGETTO FINALE)\\documentazione\\cpi.xlsx")
cpi.head(1)

In [ ]:
forecasts = []
max_cpi = 10000 # Setto un valore massimale per evitare previsioni 

for area in cpi['area_macro'].unique():
    for cat in cpi['cat_code'].unique():
        
        _cpi = cpi[(cpi['area_macro'] == area) & (cpi['cat_code'] == cat)][['year', 'cpi','cat_code']].copy()
        _cpi.rename(columns={'year': 'ds', 'cpi': 'y'}, inplace=True)
        _cpi['cap'] = max_cpi
        _cpi['ds'] = pd.to_datetime(_cpi['ds'], format='%Ye')
        
        if len(_cpi) < 2 or _cpi['y'].notna().sum() < 2:
          continue
        
        # Creo e addestro il modello
        model = Prophet(growth='logistic', yearly_seasonality=True, changepoint_prior_scale=0.05)
        model.fit(_cpi)

        # Previsione per i prossimi 7 anni
        future = model.make_future_dataframe(periods=7, freq='Y')
        future['cap'] = max_cpi
        
        forecast = model.predict(future)
    
        forecast = forecast[forecast['ds'].dt.year > _cpi['ds'].max().year] # Filtro solo gli anni superiori a quelli contenuti in cpi
    
        forecast['area_macro'] = area
        forecast['cat_code'] = cat
        
        forecasts.append(forecast[['area_macro', 'ds', 'yhat', 'cat_code']])   

cpi_forecast = pd.concat(forecasts)

In [ ]:
cpi_forecast.rename(columns={'ds': 'year', 'yhat': 'cpi'}, inplace=True)

In [ ]:
cpi_forecast.head(50)

In [ ]:
cpi_forecast.to_csv("C:\\Users\\vladb\\OneDrive\\Desktop\\ANALISI DATA\\EPICODE\\MODULO 6 (PROGETTO FINALE)\\documentazione\\cpi_forecast.csv", index=False)

In [ ]:
wages = pd.read_excel("C:\\Users\\vladb\\OneDrive\\Desktop\\ANALISI DATA\\EPICODE\\MODULO 6 (PROGETTO FINALE)\\documentazione\\wages.xlsx")
wages.head(1)

In [ ]:
_forecasts = []

for state in wages['area'].unique():
    for occ in wages['occ_code'].unique():
        
        _wages = wages[(wages['area'] == state) & (wages['occ_code'] == occ)][['year', 'a_median', 'occ_code', 'tot_emp']].copy()
        _wages.rename(columns={'year': 'ds', 'a_median': 'y'}, inplace=True)
        _wages['ds'] = pd.to_datetime(_wages['ds'], format='%Ye')
        
        if len(_wages) < 2 or _wages['y'].notna().sum() < 2:
          continue
        
        # Creo e addestro il modello
        model = Prophet(yearly_seasonality=True)
        model.add_regressor('tot_emp')
        model.fit(_wages)

        # Previsione per i prossimi 7 anni
        future = model.make_future_dataframe(periods=7, freq='Y')
        
        _forecast = model.predict(future)
    
        _forecast = _forecast[_forecast['ds'].dt.year > _wages['ds'].max().year] # Filtro solo gli anni superiori a quelli contenuti in cpi
    
        _forecast['area'] = state
        _forecast['occ_code'] = occ
        
        _forecasts.append(_forecast[['area', 'occ_code', 'tot_emp', 'yhat', 'ds']])   

wages_forecast = pd.concat(_forecasts)

In [ ]:
_forecasts = []

for state in wages['area'].unique():
    for occ in wages['occ_code'].unique():
        
        df = wages[(wages['area'] == state) & (wages['occ_code'] == occ)].copy()
        
        if len(df) < 2:
            continue

        df['ds'] = pd.to_datetime(df['year'], format='%Ye')

        ### MODELLO 1: a_median ###
        if df['a_median'].notna().sum() >= 2:
            m1 = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
            m1.fit(df[['ds', 'a_median']].rename(columns={'a_median': 'y'}))
            future1 = m1.make_future_dataframe(periods=7, freq='Y')
            forecast1 = m1.predict(future1)
            forecast1 = forecast1[['ds', 'yhat']].rename(columns={'yhat': 'a_median'})
        else:
            continue

        ### MODELLO 2: tot_emp ###
        if df['tot_emp'].notna().sum() >= 2:
            m2 = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
            m2.fit(df[['ds', 'tot_emp']].rename(columns={'tot_emp': 'y'}))
            future2 = m2.make_future_dataframe(periods=7, freq='Y')
            forecast2 = m2.predict(future2)
            forecast2 = forecast2[['ds', 'yhat']].rename(columns={'yhat': 'tot_emp'})
        else:
            continue

        ### MERGE previsioni ###
        merged = pd.merge(forecast1, forecast2, on='ds')
        merged = merged[merged['ds'].dt.year > df['ds'].dt.year.max()]

        merged['area'] = state
        merged['occ_code'] = occ
        merged['year'] = merged['ds'].dt.year

        _forecasts.append(merged[['area', 'occ_code', 'year', 'a_median', 'tot_emp']])

wages_forecast = pd.concat(_forecasts)


In [ ]:
wages_forecast.head(50)

In [ ]:
wages_forecast.to_csv("C:\\Users\\vladb\\OneDrive\\Desktop\\ANALISI DATA\\EPICODE\\MODULO 6 (PROGETTO FINALE)\\documentazione\\wages_forecast.csv", index=False)